In [1]:
### IMPORTS
import math
import pandas as pd
import ScheduleChecker as sc  
import kpi                    
###


### CONSTANTS ###

GARAGE = "ehvgar"
KWH_PER_KM = 1.2      # calculated from distance matrix and the plan consumption
IDLE_ENERGY = 5.0     # how much energy a bus consumes while idling
CHARGE_RATE = 7.5     # kWH per minute, garage only

CAPACITY = 450        # idk what the real number is
SOH = (0.85, 0.95) 
SAFETY_MARGIN = 0.10
CHARGE_TARGET = 0.90  # leave the garage at 90 % of capacity (Q)

Q = CAPACITY * min(SOH)
S_MIN = Q * SAFETY_MARGIN
TARGET = CHARGE_TARGET * Q

###           ###



def load_trips(timetable,distance_matrix):
    trips = kpi.merge_route_plan(timetable, distance_matrix)
    trips['departure_time'] = sc.convert_time_to_minutes(trips['departure_time'])
    trips['departure_time'] = sc.adjust_for_midnight(trips['departure_time'])
    trips['arrival_time'] = trips['departure_time'] + trips['max_travel_time']
    trips['energy'] =  trips['distance_m']/1000 * KWH_PER_KM
    trips.sort_values(by=['departure_time'], inplace=True)
    trips.reset_index(drop=True, inplace=True)

    return trips

def build_bus(bus, remaining):
    loc, t, charge, = GARAGE, 0, Q
    rows = []
    return rows

tt = pd.read_excel("Timetable.xlsx")
dm = pd.read_excel("DistanceMatrix.xlsx")

trips = load_trips(tt,dm)
remaining_service = trips.to_dict('records')

def load_deadhead(dm):
    non_service = dm[dm['line'].isna()]
    dh = {}
    for row in non_service.itertuples():
        key = (row.start, row.end)
        value = (row.max_travel_time, row.distance_m/1000 * KWH_PER_KM)
        dh[key] = value
    return dh

def deadhead(dh, start, end):
    if start == end:
        return (0, 0.0)          # already there: no minutes, no energy
    return dh[(start, end)]

dh = load_deadhead(dm) #deadhead lookup 

def find_next_trip(remaining,dh, loc, t, charge): 

    # remaining = trips that have not been assigned to a bus yet
    # dh = deadhead lookup table
    # loc = current location of the bus
    # t = current time in minutes for the bus
    # charge = current charge in kWh for the bus

    time_ok = False

    for trip in remaining:

        

        drive_min, drive_kwh = deadhead(dh, loc, trip['start']) #lookup values

        if t + drive_min > trip['departure_time']:
            continue

        time_ok = True

        # charge calculations

        is_early = t + drive_min < trip['departure_time'] # Bool

        if is_early:
            idle_kwh = IDLE_ENERGY
        else:
            idle_kwh = 0.0

        home_kwh = deadhead(dh, trip['end'], GARAGE)[1] # from trip end to garage, only return kWh

        left = charge - drive_kwh - idle_kwh - home_kwh - trip['energy'] 
        # en
        if left < S_MIN:
            continue

        return trip, "ok"
    return None, "low battery" if time_ok else "no_trips"

        

print(remaining_service[0])




FileNotFoundError: [Errno 2] No such file or directory: 'GInasfolder/Timetable.xlsx'